In [2]:
import numpy as np
import joblib
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, confusion_matrix, classification_report
)


In [4]:
# 1. Load data
data = load_iris()
X, y = data.data, data.target

In [5]:
# 2. Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
#model without Cross-Validation
model_simple = RandomForestClassifier(random_state=42)
model_simple.fit(X_train, y_train)

y_pred = model_simple.predict(X_test)

In [7]:
#metrics without Cross-Validation
print("=== Метрики без кросс-валидации ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='macro'))
print("Recall:", recall_score(y_test, y_pred, average='macro'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))

=== Метрики без кросс-валидации ===
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
Confusion Matrix:
 [[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [16]:
# === Accuracy ===
cv_accuracy = cross_val_score(
    model_simple, X, y, cv=5, scoring='accuracy'
)
print("=== Accuracy (CV) ===")
print("Scores:", cv_accuracy)
print("Mean:", np.mean(cv_accuracy), "\n")

# === Precision (macro) ===
cv_precision = cross_val_score(
    model_simple, X, y, cv=5, scoring='precision_macro'
)
print("=== Precision Macro (CV) ===")
print("Scores:", cv_precision)
print("Mean:", np.mean(cv_precision), "\n")

# === Recall (macro) ===
cv_recall = cross_val_score(
    model_simple, X, y, cv=5, scoring='recall_macro'
)
print("=== Recall Macro (CV) ===")
print("Scores:", cv_recall)
print("Mean:", np.mean(cv_recall), "\n")

# === F1-score (macro) ===
cv_f1 = cross_val_score(
    model_simple, X, y, cv=5, scoring='f1_macro'
)
print("=== F1 Macro (CV) ===")
print("Scores:", cv_f1)
print("Mean:", np.mean(cv_f1))

=== Accuracy (CV) ===
Scores: [0.96666667 0.96666667 0.93333333 0.96666667 1.        ]
Mean: 0.9666666666666668 

=== Precision Macro (CV) ===
Scores: [0.96969697 0.96969697 0.94444444 0.96969697 1.        ]
Mean: 0.9707070707070707 

=== Recall Macro (CV) ===
Scores: [0.96666667 0.96666667 0.93333333 0.96666667 1.        ]
Mean: 0.9666666666666668 

=== F1 Macro (CV) ===
Scores: [0.96658312 0.96658312 0.93265993 0.96658312 1.        ]
Mean: 0.9664818612187034


Без кросс-валидации модель показывает идеальные метрики (1.0). Это может происходить из-за удачного разбиения данных, когда обучающая и тестовая выборки оказываются слишком похожими. В таком случае метрики могут быть переоценены и не отражать реального качества модели. При использовании кросс-валидации модель проверяется на пяти разных разбиениях. В некоторых фолдах метрики уже не идеальны, и средние значения находятся в диапазоне 0.96–0.97. Эти значения более стабильны и точнее отображают реальную способность модели обобщать данные. Таким образом, метрики кросс-валидации являются более надёжными, чем результаты одного разбиения без кросс-валидации.

In [17]:
#GridSearch 
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 3, 5, 10],
    'min_samples_split': [2, 4]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("=== Лучшие параметры ===")
print(grid.best_params_)

best_model = grid.best_estimator_


=== Лучшие параметры ===
{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}


In [18]:
#metrics after GridSearch
y_pred_gs = best_model.predict(X_test)

print("=== Метрики после GridSearch ===")
print("Accuracy:", accuracy_score(y_test, y_pred_gs))
print("Precision:", precision_score(y_test, y_pred_gs, average='macro'))
print("Recall:", recall_score(y_test, y_pred_gs, average='macro'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_gs))
print("\nClassification report:\n", classification_report(y_test, y_pred_gs))


=== Метрики после GridSearch ===
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
Confusion Matrix:
 [[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [22]:
metrics = {
    "Accuracy": "accuracy",
    "Precision Macro": "precision_macro",
    "Recall Macro": "recall_macro",
    "F1 Macro": "f1_macro"
}

for name, scoring in metrics.items():
    cv_scores = cross_val_score(best_model, X, y, cv=5, scoring=scoring)

    print(f"\n=== {name} (CV) ===")
    print("Scores:", cv_scores)
    print("Mean:", np.mean(cv_scores))


=== Accuracy (CV) ===
Scores: [0.96666667 0.96666667 0.93333333 0.96666667 1.        ]
Mean: 0.9666666666666668

=== Precision Macro (CV) ===
Scores: [0.96969697 0.96969697 0.94444444 0.96969697 1.        ]
Mean: 0.9707070707070707

=== Recall Macro (CV) ===
Scores: [0.96666667 0.96666667 0.93333333 0.96666667 1.        ]
Mean: 0.9666666666666668

=== F1 Macro (CV) ===
Scores: [0.96658312 0.96658312 0.93265993 0.96658312 1.        ]
Mean: 0.9664818612187034


После применения GridSearch качество модели практически не изменилось. Это ожидаемо, потому что датасет Iris простой, хорошо разделимый и даже базовые параметры RandomForest дают почти идеальные результаты. Кросс-валидация до и после GridSearch показывает одинаковые средние метрики (~0.96–0.97), что означает: оптимизация гиперпараметров почти не влияет на качество, так как модель и так прекрасно справляется с задачей.

In [27]:
#save the model 
joblib.dump(best_model, "../models/iris_model_best.pkl")
print("Model saved to iris_model_best.pkl")

Model saved to iris_model_best.pkl
